In [1]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import feel_it 
import google_play_scraper as gps
from google_play_scraper import Sort , reviews as google_reviews
import sqlalchemy

In [2]:
USER = "root"
PASSWORD = "Matteo00" # Inserisci la tua password di MySQL
HOST = "localhost"
DATABASE = "progetto_banca" # Il nome dello schema creato su Workbench
# Crea la connessione al database
engine = sqlalchemy.create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DATABASE}')

In [3]:
df = pd.read_sql_query("SELECT * FROM progetto_banca", engine)

In [4]:
from transformers import pipeline
classifier = pipeline("text-classification",model="neuraly/bert-base-italian-cased-sentiment",)



config.json: 0.00B [00:00, ?B/s]

c:\Users\matte\anaconda.3\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\matte\.cache\huggingface\hub\models--neuraly--bert-base-italian-cased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

Device set to use cuda:0


In [7]:
import torch
if torch.cuda.is_available():
    device = -1


In [8]:
from tqdm.auto import tqdm

df = df = pd.read_sql_query("SELECT * FROM progetto_banca", engine)
df = df.dropna(subset=['testo']).reset_index(drop=True)


if torch.cuda.is_available():
    device = 0
    
else:
    device = -1
    

sentiment_pipeline = pipeline(
    "text-classification",
    model="neuraly/bert-base-italian-cased-sentiment",
    device=device
)


testi = df['testo'].tolist()
batch_size = 128
risultati = []


for output in tqdm(sentiment_pipeline(testi, batch_size=batch_size, truncation=True, max_length=512), total=len(testi)):
    risultati.append(output)

# 5. Salvataggio risultati
df['sentiment_feelit'] = [res['label'] for res in risultati]
df['sentiment_score'] = [round(res['score'], 4) for res in risultati]



Device set to use cuda:0


  0%|          | 0/77852 [00:00<?, ?it/s]

In [9]:
df.to_sql('progetto_banca_sentiment', con=engine, if_exists='replace', index=False)

77852

In [10]:

df_completo = pd.read_sql_query("SELECT * FROM progetto_banca_sentiment", engine)
df_negative = df_completo[df_completo['sentiment_feelit'] == 'negative'].copy()
nome_file_nlp = 'sentiment_negativo'
df_negative.to_sql(nome_file_nlp, con=engine, if_exists='replace', index=False)

22201

In [11]:
import spacy
df_neg = pd.read_sql_query("SELECT * FROM sentiment_negativo", engine)
nlp = spacy.load("it_core_news_sm", disable=["ner", "parser"])
testo_lemmatizzato = []
lemmi_totali = []
bigrammi_totali = []
for doc in nlp.pipe(df_neg['testo'], batch_size=50):
    lemmi = [t.lemma_.lower() for t in doc if t.is_alpha and not t.is_stop and len(t.text) > 2]
    
    testo_lemmatizzato.append(" ".join(lemmi))
    lemmi_totali.append(", ".join(lemmi))
    
    bigrams = [f"{lemmi[i]} {lemmi[i+1]}" for i in range(len(lemmi) - 1)] if len(lemmi) >= 2 else []
    bigrammi_totali.append(", ".join(bigrams))

df_neg['testo_lemmatizzato'] = testo_lemmatizzato
df_neg['lemmi_totali'] = lemmi_totali
df_neg['bigrammi_totali'] = bigrammi_totali
df_neg.to_sql('progetto_banca_defsql', con=engine, if_exists='replace', index=False)

22201

In [ ]:
 from collections import Counter

df_top_lemmi_bigrammi=pd.read_sql_query("select * from progetto_banca_defsql", engine)
lemmi_totali = [lemma.strip() for riga in df['lemmi_totali'].dropna() for lemma in str(riga).split(',')]
bigrammi_totali = [bigramma.strip() for riga in df['bigrammi_totali'].dropna() for bigramma in str(riga).split(',')]
lemmi_top = []
bigrammi_top = []
conteggio_lemmi = Counter(lemmi_totali)
conteggio_bigrammi = Counter(bigrammi_totali)

for lemma, freq in conteggio_lemmi.most_common(20):

    lemmi_top.append(f"{lemma}: {freq}")

for bigramma, freq in conteggio_bigrammi.most_common(20):

    bigrammi_top.append(f"{bigramma}: {freq}")

df_lemmi = pd.DataFrame(conteggio_lemmi.most_common(20), columns=['parola_chiave', 'frequenza'])
df_lemmi['tipo'] = 'lemma'
df_bigrammi = pd.DataFrame(conteggio_bigrammi.most_common(20), columns=['parola_chiave', 'frequenza'])
df_bigrammi['tipo'] = 'bigramma'
# 4. Unione in un'unica tabella dizionario
df_top_words = pd.concat([df_lemmi, df_bigrammi], ignore_index=True)
# 5. Salvataggio su MySQL in una NUOVA tabella dedicata
nome_tabella_top = 'top_20_lemmi_frequenti'
df_top_words.to_sql(nome_tabella_top, con=engine, if_exists='replace', index=False)
# Stampa di controllo
print(f"Tabella '{nome_tabella_top}' generata e salvata su MySQL con successo.")
print(df_top_words.head()) 

Tabella 'top_20_lemmi_frequenti' generata e salvata su MySQL con successo.
   parola_chiave  frequenza   tipo
0            app      11024  lemma
1     funzionare       4592  lemma
2       problema       4091  lemma
3  aggiornamento       3098  lemma
4          conto       2752  lemma
